In [1]:
pip install statsmodels numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install celer

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
import pandas as pd
from statsmodels.nonparametric.kernel_regression import KernelReg
from celer import GroupLasso  # <-- Import the super-fast 'celer' engine!

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
import joblib

In [5]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

In [6]:
df_encoded = pd.read_csv("df_cleaned.csv")

In [7]:
# 1️⃣ Create unscaled copy
df_unscaled = df_encoded.copy()

In [8]:
df_unscaled

,LOCATION,PRICE,ROOMS,BATHROOMS,CAR_PARKS,PROPERTY_BASE,PROPERTY_TYPE,BUILT-UP,FURNISHING _STATUS,PRICE_LOG,BUILT-UP_LOG
0,SETAPAK,200000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,800.0,UNFURNISHED,12.206078,6.685861
1,SETAPAK,210000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,802.0,PARTLY FURNISHED,12.254868,6.688355
2,SETAPAK,215000.0,3.0,2.0,1.0,CONDOMINIUM,CORNER,900.0,PARTLY FURNISHED,12.278398,6.803505
3,SETAPAK,215000.0,3.0,2.0,1.0,CONDOMINIUM,INTERMEDIATE,800.0,UNFURNISHED,12.278398,6.685861
4,SETAPAK,220000.0,3.0,2.0,1.0,CONDOMINIUM,CORNER,832.0,PARTLY FURNISHED,12.301387,6.725034
...,...,...,...,...,...,...,...,...,...,...,...
21553,BANGSAR,16500000.0,6.0,7.0,5.0,CONDOMINIUM,OTHER,6900.0,PARTLY FURNISHED,16.618871,8.839422
21554,MONT KIARA,17500000.0,5.0,3.0,3.0,CONDOMINIUM,OTHER,1830.0,PARTLY FURNISHED,16.677711,7.512618
21555,KLCC,18000000.0,5.0,6.0,4.0,CONDOMINIUM,PENTHOUSE,10904.0,FULLY FURNISHED,16.705882,9.296977
21556,KLCC,23300000.0,7.0,5.0,3.0,CONDOMINIUM,OTHER,7200.0,PARTLY FURNISHED,16.963964,8.881975


In [9]:
# 2️⃣ Identify features to scale (exclude target column)
features_to_scale = ['ROOMS', 'BATHROOMS', 'CAR_PARKS', 'BUILT-UP_LOG']

In [10]:
# 3️⃣ Initialize scaler
scaler = StandardScaler()

In [11]:
# 4️⃣ Create scaled version
df_scaled = df_encoded.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df_encoded[features_to_scale])

In [12]:
print(df_scaled.columns)

Index(['LOCATION', 'PRICE', 'ROOMS', 'BATHROOMS', 'CAR_PARKS', 'PROPERTY_BASE',
       'PROPERTY_TYPE', 'BUILT-UP', 'FURNISHING _STATUS', 'PRICE_LOG',
       'BUILT-UP_LOG'],
      dtype='object')


In [13]:
# 5️⃣ Display scaling results
print("\n✅ Feature scaling complete!")
print("Mean of scaled features (≈0):")
print(df_scaled[features_to_scale].mean().round(2).head())

print("\nStandard deviation (≈1):")
print(df_scaled[features_to_scale].std().round(2).head())


✅ Feature scaling complete!
Mean of scaled features (≈0):
ROOMS           0.0
BATHROOMS      -0.0
CAR_PARKS       0.0
BUILT-UP_LOG    0.0
dtype: float64

Standard deviation (≈1):
ROOMS           1.0
BATHROOMS       1.0
CAR_PARKS       1.0
BUILT-UP_LOG    1.0
dtype: float64


In [16]:
# --- Scaled version ---
X_scaled = df_scaled.drop(columns=["price_myr"], errors="ignore")
y_unscaled = df_unscaled["price_myr"]

X_train_scaled, X_test_scaled, y_train_scaled, y_test_unscaled = train_test_split(
    X_scaled, y_unscaled, test_size=0.2, random_state=42
)

# --- K-Fold setup ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n✅ Data split complete!")
print(f"Scaled:   {X_train_scaled.shape} train, {X_test_scaled.shape} test")


✅ Data split complete!
Scaled:   (17712, 65) train, (4429, 65) test


In [17]:
models = {
    "Linear Regression": (LinearRegression(), {}),  # No hyperparameters to tune
    "Lasso Regression": (Lasso(random_state=42, max_iter=3000), {
        "alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 20.0, 30.0, 100.0],  # Hyperparameter grid for alpha
    })
}

In [27]:
selected_columns = X.iloc[:, 5:56]

NameError: name 'X' is not defined

In [ ]:
# 1. ⚡ SET UP CELER GROUP INDICES!
# celer takes a list of lists, where each sub-list contains the column index numbers for that group.
# EXAMPLE for 8 columns: 
# Group 0 (Structure): cols 0, 1, 2
# Group 1 (Location): cols 3, 4
# Group 2 (Economics): cols 5, 6, 7
celer_groups = [[0, 1, 2], [3, 4], [5, 6, 7]]

In [18]:
print("✅ Models and parameter grids prepared.")

✅ Models and parameter grids prepared.


In [19]:
results = []
trained_models = {}  # Store fitted models

for name, (model, param_grid) in models.items():
    print(f"\n🚀 Training model: {name}")
    
    # 1. ⚡ FIX: Use Scaled Features (X), but keep Target Prices (y) in unscaled RM!
    # (Assuming you have y_train_unscaled and y_test_unscaled from your earlier split)
    X_train, y_train = X_train_scaled, y_train_scaled
    X_test, y_test = X_test_scaled, y_test_scaled
    
    # 2. ⚡ FIX: Check if the model has a param_grid. If yes, run K-Fold GridSearchCV!
    if param_grid:
        grid = GridSearchCV(
            estimator=model, 
            param_grid=param_grid, 
            cv=kf,  # <-- Connects your 5-fold KFold setup!
            scoring="neg_mean_squared_error", 
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        model = grid.best_estimator_  # Grab the winning Lasso model
        print(f"🔧 Best Params: {grid.best_params_}")
    else:
        # Linear Regression has no grid, so just fit directly
        model.fit(X_train, y_train)

    trained_models[name] = model  # Save trained version

    # 3. TESTING / EVALUATION PHASE (Now evaluating in real RM!)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R²": r2})
    print(f"{name} → RMSE: RM {rmse:,.2f}, MAE: RM {mae:,.2f}, R²: {r2:.3f}")


🚀 Training model: Linear Regression
Linear Regression → RMSE: RM 241,851.05, MAE: RM 169,524.25, R²: 0.680

🚀 Training model: Lasso Regression
🔧 Best Params: {'alpha': 100.0}
Lasso Regression → RMSE: RM 241,839.85, MAE: RM 169,495.71, R²: 0.680


In [21]:
# --- 5️⃣ Compare Model Results ---
eval_df = pd.DataFrame(results).sort_values("RMSE")
print("\n🏁 MODEL PERFORMANCE SUMMARY:")
print(eval_df)

best_model_name = eval_df.iloc[0]["Model"]
print(f"\n🎯 Best Model: {best_model_name}")
best_model = trained_models[best_model_name]


🏁 MODEL PERFORMANCE SUMMARY:
               Model           RMSE            MAE        R²
1   Lasso Regression  241839.847183  169495.714824  0.680319
0  Linear Regression  241851.046209  169524.254726  0.680290

🎯 Best Model: Lasso Regression
